In [1]:
import xarray as xr
import json
import yaml

In [2]:
adf_loc = "/glade/u/home/qingyuany/ADF/"
obs_loc = "/glade/campaign/cgd/amp/amwg/ADF_obs/"

In [3]:
with open(adf_loc + "lib/adf_variable_defaults.yaml", "r") as f:
    obs_info = yaml.safe_load(f)

In [4]:
obs_info['FSNTC']

{'category': 'TOA energy flux',
 'obs_file': 'CERES_EBAF_Ed4.1_2001-2020.nc',
 'obs_name': 'CERES_EBAF_Ed4.1',
 'obs_var_name': 'toa_sw_clr_t_mon',
 'pct_diff_contour_levels': [-100,
  -75,
  -50,
  -40,
  -30,
  -20,
  -10,
  -8,
  -6,
  -4,
  -2,
  0,
  2,
  4,
  6,
  8,
  10,
  20,
  30,
  40,
  50,
  75,
  100],
 'pct_diff_colormap': 'PuOr_r'}

In [5]:
obs_info["FLUTC"]

{'obs_file': 'CERES_EBAF_Ed4.1_2001-2020.nc',
 'obs_name': 'CERES_EBAF_Ed4.1',
 'obs_var_name': 'toa_lw_clr_t_mon'}

In [6]:
voi = ['RESTOM', 'FSNT', 'FLNT', 'SWCF', 'LWCF', 'PRECT', 'TGCLDLWP', 'FLUTC', 'FSNTC', 'TMQ', 'LHFLX', 'FSNTC']


In [7]:
camoutput = xr.open_dataset("/glade/work/qingyuany/cam7_re/v0/ppe_out.nc")

In [11]:
var_obs_dict = {}

for v in voi:
    t_where = obs_info[v]['obs_file']
    t_var_nm = obs_info[v]['obs_var_name']
    print("{:<10} {:<50} {:<10}".format(v, t_where, t_var_nm))
    var_obs_dict[v] = t_var_nm

RESTOM     CERES_EBAF_Ed4.1_2001-2020.nc                      toa_net_all_mon
FSNT       CERES_EBAF_Ed4.1_2001-2020.nc                      fsnt      
FLNT       CERES_EBAF_Ed4.1_2001-2020.nc                      toa_lw_all_mon
SWCF       CERES_EBAF_Ed4.1_2001-2020.nc                      toa_cre_sw_mon
LWCF       CERES_EBAF_Ed4.1_2001-2020.nc                      toa_cre_lw_mon
PRECT      ERAI_all_climo.nc                                  PRECT     
TGCLDLWP   TGCLDLWP_ERA5_monthly_climo_197901-202112.nc       TGCLDLWP  
FLUTC      CERES_EBAF_Ed4.1_2001-2020.nc                      toa_lw_clr_t_mon
FSNTC      CERES_EBAF_Ed4.1_2001-2020.nc                      toa_sw_clr_t_mon
TMQ        ERAI_all_climo.nc                                  PREH2O    
LHFLX      ERAI_all_climo.nc                                  LHFLX     
FSNTC      CERES_EBAF_Ed4.1_2001-2020.nc                      toa_sw_clr_t_mon


In [12]:
obs = []
for v in voi:
    if v == 'FSNTC':
        a  = xr.open_dataset(obs_loc + obs_info[v]["obs_file"])[obs_info[v]['obs_var_name']]
        b = xr.open_dataset(obs_loc + obs_info[v]["obs_file"])['solar_mon']
        temp_v = b -a 
        temp_v.name = 'toa_sw_clr_t_mon'
    else:
        temp_v = xr.open_dataset(obs_loc + obs_info[v]["obs_file"])[obs_info[v]['obs_var_name']]
        
        
    temp_v = temp_v.interp(lat=camoutput.lat, lon=camoutput.lon, method="nearest")
    print(temp_v.shape)

    obs.append(temp_v.mean(dim = "time"))

(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)
(12, 192, 288)


In [13]:
obs_ds = xr.Dataset({da.name: da for da in obs})

In [14]:
obs_ds["PRECT"] = obs_ds["PRECT"]/(1000 * 24 * 3600)

In [15]:
obs_ds.to_netcdf("/glade/work/qingyuany/cam7_re/obs.nc")